## 📁 Unified Billing Pipeline Execution (`InvoiceHeader_df`)

This section handles the end-to-end ingestion, processing, and unification of all corporate billing streams. To build a single, comprehensive source of truth for downstream reporting, the pipeline normalizes **Sales Invoices** and **Credit Memos** into an identical schema format before combining them.

### ⚙️ Pipeline Process Flow

1. **Sales Invoice Engineering**
   * Ingests the baseline `bz_salesinvoices` bronze dataset.
   * Runs validation checks to capture duplicates or null primary keys, routing records with issues directly into the `exception_table`.
   * Standardizes fields to conform to the silver layer's master billing layout, setting the transaction identifier type strictly to **`Inv`**.

2. **Credit Memo Engineering**
   * Ingests the `bz_salescreditmemos` bronze dataset.
   * Utilizes identical row-ranking deduplication and primary key null checks to isolate pipeline exceptions.
   * Normalizes credit values (e.g., multiplying totals by `-1` to ensure correct financial balancing) and sets the transaction identifier type strictly to **`CR`**.

3. **Master Schema Alignment & Reference Joins**
   * Discards unneeded structural variations and maps both dataframes to matching enterprise data types.
   * Performs contextual left-joins against master dimensions (such as `sl_salesrep`) to resolve human-readable entity references inline.

4. **Data Unification (`unionByName`)**
   * Combions both normalized operational data streams into a single dataset.
   * Resolves columns by name to protect against upstream source shifts, generating the unified global dataset: **`invoice_header_df`**.

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, trim, row_number, current_date, current_timestamp, sha2, concat_ws
from pyspark.sql.window import Window
import urllib

# ==========================================
# 1. PARAMETERIZATION & CONFIGURATION
# ==========================================
# Senior engineers NEVER hardcode paths or variables. We use widgets.
dbutils.widgets.text("company_key", "ABC", "Company Key")
dbutils.widgets.text("brand_key", "ABC", "Brand Key")
dbutils.widgets.text("catalog_name", "erp_lakehouse", "Catalog Name")
dbutils.widgets.text("schema_name", "silver", "Schema Name")
dbutils.widgets.text("silver_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/sl_invoiceheader", "Silver External Path")
dbutils.widgets.text("exception_external_path", "abfss://silver@bbmanufacturingprod.dfs.core.windows.net/delta/exception_table", "Exception Table Path")


COMPANY_KEY = dbutils.widgets.get("company_key")
BRAND_KEY = dbutils.widgets.get("brand_key")
CATALOG_NAME= dbutils.widgets.get("catalog_name")
SILVER_SCHEMA= dbutils.widgets.get("schema_name")
SILVER_PATH = dbutils.widgets.get("silver_external_path")
EXCEPTION_PATH = dbutils.widgets.get("exception_external_path")



# Processing SalesInvoices table

In [0]:
# ==========================================
# 2. DATA LOADING (BRONZE LAYER)
# ==========================================
print("Reading data from Bronze Delta Table...")
bronze_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_salesinvoices")

# Inject control columns early
enriched_df = bronze_df \
    .withColumn('RecordStatus', lit('0')) \
    .withColumn('CompanyKey', lit(COMPANY_KEY)) \
    .withColumn('BrandKey', lit(BRAND_KEY))


In [0]:
from pyspark.sql.functions import col, when, trim, row_number
from pyspark.sql.window import Window

# ==========================================
# 3. DATA QUALITY & INTEGRITY RULES (SILVER AUDIT)
# ==========================================

print("Executing pre-shuffle Validation Rule 1: Multi-Column Null/Blank Check... 🔍")

# If ANY of these key identifiers are missing, flag it as '2' straight away.
validated_df = enriched_df.withColumn(
    "RecordStatus",
    when(
        (col("id").isNull()) | (trim(col("id")) == "") |
        (col("invoiceDate").isNull()) | (trim(col("invoiceDate")) == "") |
        (col("Number").isNull()) | (trim(col("Number")) == ""),
        lit('2')
    ).otherwise(col("RecordStatus"))
)


print("Executing Validation Rule 2: Performance-Optimized Deduplication... 🚀")

# Never order by the ID itself; order by a timestamp to ensure you keep the LATEST operational record.
window_spec = Window.partitionBy("id", "CompanyKey", "BrandKey").orderBy(col("ingestion_time").desc())

# Apply the window ranking function defensively
processed_df = validated_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn(
        "RecordStatus", 
        # CRITICAL CONTEXT: Only flag as a duplicate ('1') if it passed the previous check and is still '0'
        when((col("RecordStatus") == '0') & (col("row_num") > 1), lit('1')).otherwise(col("RecordStatus"))
    ) \
    .drop("row_num")


# Persist to memory to freeze execution graph before splitting into forks (Exceptions vs Clean Data)
processed_df.cache()

print("Data quality auditing complete. Frame cached successfully.")
display(processed_df.limit(10))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, trim, array, explode, when, current_date, current_timestamp, sha2, concat_ws, coalesce
from pyspark.sql.utils import AnalysisException

print("Processing and routing complex multi-column data exceptions... 🚀")

# Ensure the dedicated silver schema isolation layer exists natively
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Ensure your global exception storage structure is instantiated
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table (
    ExceptionID STRING,
    BrandKey STRING,
    RecordKey STRING,
    TableName STRING,
    ColumnName STRING,
    ExceptionDetails STRING,
    SysCreatedDate DATE,
    SysCreatedBy STRING
)
USING DELTA
LOCATION '{EXCEPTION_PATH}'
""")

# ==========================================
# 1. FORK A: PROCESS DUPLICATE RECORDS (Status '1')
# ==========================================
duplicate_records_df = processed_df.filter(col("RecordStatus") == '1')

duplicate_exceptions = duplicate_records_df.select(
    # Stable, deterministic cryptographic unique identifier
    sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    # Safely handle potential nulls in ID during string mapping
    coalesce(col("id").cast("string"), lit("UNKNOWN_ID")).alias("RecordKey"),
    lit("bz_SalesInvoices").alias("TableName"),
    lit("").alias("ColumnName"),
    lit("Duplicate record identified during delta ranking window").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
)

# ==========================================
# 2. FORK B: PROCESS NULL/BLANK RECORDS (Status '2')
# ==========================================
null_blank_records_df = processed_df.filter(col("RecordStatus") == '2')

# Dynamically construct array elements for validation tracking
columns_to_check = ["id", "invoiceDate", "Number"]
exprs = [
    when((col(c).isNull()) | (trim(col(c)) == ""), lit(c)).otherwise(lit(None))
    for c in columns_to_check
]

# Pack errors into an array structure
flagged_nulls_df = null_blank_records_df.withColumn("NullColumns", array(*exprs))

# Explode elements to unpack one audit log entry per failing column
null_exceptions = flagged_nulls_df.select(
    sha2(concat_ws("||", col("Number"), col("invoiceDate"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    # Senior Strategy: If id is null, fall back to showing the invoice Number so business can search it!
    coalesce(col("id").cast("string"), col("Number").cast("string"), lit("UNKNOWN_ROW")).alias("RecordKey"),
    lit("bz_SalesInvoices").alias("TableName"),
    explode(col("NullColumns")).alias("ColumnName"),
    lit("Null/Blank primary value restriction violation").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
).filter(col("ColumnName").isNotNull()) # Drop rows for columns that actually passed checks

# ==========================================
# 3. CONSOLIDATE AND APPEND TO AUDIT STORE
# ==========================================
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)

# Cache checkpoint to optimize action evaluation
final_exceptions_df.cache()

if final_exceptions_df.count() > 0:
    print(f"Writing exceptions to log repository: {CATALOG_NAME}.{SILVER_SCHEMA}.exception_table")
    # Write cleanly using modern Delta formatting API
    final_exceptions_df.write.format("delta").mode("append").saveAsTable("erp_lakehouse.silver.exception_table")

# Remove the exceptions from your main pipeline dataframe to continue clean processing downstream
clean_result_df = processed_df.filter(~col("RecordStatus").isin(['1', '2']))

# Clear exception tracking from cluster memory cache overhead
final_exceptions_df.unpersist()

print("Exception processing routine complete. Clean corporate records isolated successfully! 🏁")

In [0]:
# ==========================================
# 5. TRANSFORM CLEAN DATA FOR SILVER LAYER
# ==========================================

from pyspark.sql.functions import coalesce, col, lit, when

print("Transforming valid corporate records...")
clean_records_df = processed_df.filter(col("RecordStatus") == '0')

print("Transforming corporate Product Line records with null elimination... 🚀")

from pyspark.sql.functions import lit, col, when, datediff, coalesce

final_Inv_df = clean_records_df.select(
    # Core Dimension Keys & Source IDs
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("id").alias("InvSrcId").cast("string"),
    coalesce(col("salesperson"), lit("")).alias("SalesRepInSrcId").cast("string"),
    coalesce(col("customerNumber"), lit("")).alias("CustSrcId").cast("string"),
    coalesce(col("number"), lit("")).alias("InvKey").cast("string"),
    coalesce(col("OrderNumber"), lit("")).alias("SOKey").cast("string"),
    
    # Constants for Invoice Categorization
    lit("Inv").cast("string").alias("InvType"),
    lit("Invoice").cast("string").alias("InvTypeDesc"),
    
    # Process & Operational Status Logic
    coalesce(col("status"), lit("")).alias("InvProcessStatus").cast("string"),
    
    when(col("number") == "Open", lit("Open"))
        .when(col("status").isin("Cancelled", "Canceled"), lit("Cancelled"))
        .when(col("status").isin("Paid", "Corrective"), lit("Completed"))
        .otherwise(lit("Draft")).alias("InvStatus").cast("string"),
        
    when(col("status") == "Paid", lit("Paid"))
        .when(col("status").isin("Cancelled", "Canceled"), lit("Cancelled"))
        .when((col("status") == "Open") & 
              (coalesce(col("RemainingAmount"), lit(0)) != 0) & 
              (col("totalamountincludingtax") != col("remainingamount")), lit("Partially paid"))
        .otherwise(lit("Yet to Pay")).alias("InvPaidStatus").cast("string"),
    
    # Temporal Transformations & Date Normalization
    when(col("invoiceDate") == "1900-01-01", lit(None)).otherwise(col("invoiceDate")).alias("InvDt").cast("timestamp"),
    when(col("postingDate") == "1900-01-01", lit(None)).otherwise(col("postingDate")).alias("InvPostedDt").cast("timestamp"),
    when(col("dueDate") == "1900-01-01", lit(None)).otherwise(col("dueDate")).alias("InvDueDt").cast("timestamp"),
    datediff(col("dueDate"), col("invoiceDate")).alias("InvDueDays").cast("int"),
    
    # Customer Metadata Block
    coalesce(col("salesperson"), lit("")).alias("SalesRepInKey").cast("string"),
    coalesce(col("customerNumber"), lit("")).alias("CustKey").cast("string"),
    coalesce(col("customerName"), lit("")).alias("CustName").cast("string"),
    coalesce(col("ShipToContact"), lit("")).alias("CustContactName").cast("string"),
    
    # Bill-To Address Details
    coalesce(col("billToCustomerNumber"), lit("")).alias("CustBillToAddrKey").cast("string"),
    coalesce(col("BillToName"), lit("")).alias("CustBillToName").cast("string"),
    coalesce(col("billToAddressLine1"), lit("")).alias("CustBillToAddr1").cast("string"),
    coalesce(col("billToAddressLine2"), lit("")).alias("CustBillToAddr2").cast("string"),
    coalesce(col("billToCity"), lit("")).alias("CustBillToCity").cast("string"),
    coalesce(col("billToState"), lit("")).alias("CustBillToState").cast("string"),
    coalesce(col("BilltoCountry"), lit("")).alias("CustBillToCountry").cast("string"),
    coalesce(col("billToPostCode"), lit("")).alias("CustBillToZipCd").cast("string"),
    
    # Ship-To Address Details
    coalesce(col("shipToName"), lit("")).alias("CustShipToName").cast("string"),
    coalesce(col("shipToAddressLine1"), lit("")).alias("CustShipToAddr1").cast("string"),
    coalesce(col("shipToAddressLine2"), lit("")).alias("CustShipToAddr2").cast("string"),
    coalesce(col("shipToCity"), lit("")).alias("CustShipToCity").cast("string"),
    coalesce(col("shipToState"), lit("")).alias("CustShipToState").cast("string"),
    coalesce(col("ShipToCountry"), lit("")).alias("CustShipToCountry").cast("string"),
    coalesce(col("shipToPostCode"), lit("")).alias("CustShipToZipCd").cast("string"),
    
    # Sell-To Address Details
    coalesce(col("sellToAddressLine1"), lit("")).alias("CustSellToAddr1").cast("string"),
    coalesce(col("sellToAddressLine2"), lit("")).alias("CustSellToAddr2").cast("string"),
    coalesce(col("sellToCity"), lit("")).alias("CustSellToCity").cast("string"),
    coalesce(col("sellToState"), lit("")).alias("CustSellToState").cast("string"),
    coalesce(col("sellToCountry"), lit("")).alias("CustSellToCountry").cast("string"),
    coalesce(col("sellToPostCode"), lit("")).alias("CustSellToZipCd").cast("string"),
    
    # Currency Metrics & Exchange Factors
    lit("INR").cast("string").alias("CustCurr"),
    lit("INR").cast("string").alias("LocCurr"),
    when(col("invoiceDate") == "1900-01-01", lit(None)).otherwise(col("invoiceDate")).alias("ExDt").cast("timestamp"),
    lit(1.0).cast("double").alias("ExRtLCCC"),
    lit(1.0).cast("double").alias("ExRtCCLC"),
    lit(1.0).cast("double").alias("ExRtLCIN"),
    
    # Financial Value & Tax Calculations
    coalesce(col("TotalAmountExcludingTax"), lit(0.0)).cast("double").alias("TotalAmountExcludingTax"),    
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValCC"),
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValLC"),
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValIN"),    
    # Derived Accounting Balances
    (coalesce(col("TotalAmountExcludingTax"), lit(0.0)) - coalesce(col("remainingamount"), lit(0.0))).cast("double").alias("InvPaidValCC"),
    (coalesce(col("TotalAmountExcludingTax"), lit(0.0)) - coalesce(col("remainingamount"), lit(0.0))).cast("double").alias("InvPaidValLC"),
    (coalesce(col("TotalAmountExcludingTax"), lit(0.0)) - coalesce(col("remainingamount"), lit(0.0))).cast("double").alias("InvPaidValIN"),
    
    coalesce(col("remainingamount"), lit(0.0)).cast("double").alias("InvDueValCC"),
    coalesce(col("remainingamount"), lit(0.0)).cast("double").alias("InvDueValLC"),
    coalesce(col("remainingamount"), lit(0.0)).cast("double").alias("InvDueValIN"),
    
    # System Audit Controls
    col("lastModifiedDateTime").alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)
display(final_Inv_df.limit(10))

In [0]:
# Read the reference master table for sales representatives
sales_rep_master_df = spark.read.table("erp_lakehouse.silver.sl_salesperson")

# Join, clean missing names with coalesce, and drop the join key in a single, fluid operation
invoice_df = final_Inv_df.join(
    sales_rep_master_df.select(col("SalesRepSrcId"), col("SalesRepName")),
    final_Inv_df["SalesRepInSrcId"] == sales_rep_master_df["SalesRepSrcId"],
    "left"
).withColumn(
    "SalesRepName", 
    coalesce(col("SalesRepName"), lit(""))
).drop("SalesRepSrcId")

# Single display action for validation checking
display(invoice_df.limit(10))

## Transforming Credit Memos


In [0]:

print("Starting optimized Credit Memos pipeline processing... 💳")

# ==========================================
# 1. INGESTION & DATA STAGING
# ==========================================
sales_credit_df = spark.read.table(f"{CATALOG_NAME}.bronze.bz_salescreditmemos")

# Initialize structural routing constants
added_df = sales_credit_df.withColumn('RecordStatus', lit('0')) \
                          .withColumn('CompanyKey', lit('ABC')) \
                          .withColumn('BrandKey', lit(BRAND_KEY))

In [0]:

# ==========================================
# 2. DEDUPLICATION (WINDOW FUNCTION ROUTINE)
# ==========================================
window_spec = Window.partitionBy("id", "CompanyKey", "BrandKey").orderBy("id")

# Identify duplicates cleanly using row numbers
result_df = added_df.withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("RecordStatus", when(col("row_num") > 1, lit('1')).otherwise(col("RecordStatus"))) \
    .drop("row_num")

# Identify row-level missing or blank constraints
result_df = result_df.withColumn(
    "RecordStatus",
    when((col("id").isNull()) | (trim(col("id")) == ""), lit('2')).otherwise(col("RecordStatus"))
)


In [0]:
# ==========================================
# 3. ENTERPRISE EXCEPTION HANDLING ROUTINE
# ==========================================
duplicate_records_df = result_df.filter(col("RecordStatus") == '1')
null_blank_records_df = result_df.filter(col("RecordStatus") == '2')

# Fork A: Duplicate Log Generation
duplicate_exceptions = duplicate_records_df.select(
    sha2(concat_ws("||", col("id"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    coalesce(col("id").cast("string"), lit("UNKNOWN_ID")).alias("RecordKey"),
    lit("bz_SalesCreditMemos").alias("TableName"),
    lit("").alias("ColumnName"),
    lit("Duplicate record found during delta ranking window").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
)

# Fork B: Null Constraint Log Generation (with human-readable invoice reference fallback)
null_exceptions = null_blank_records_df.select(
    sha2(concat_ws("||", col("number"), current_timestamp()), 256).alias("ExceptionID"),
    lit(BRAND_KEY).alias("BrandKey"),
    coalesce(col("id").cast("string"), col("number").cast("string"), lit("UNKNOWN_ROW")).alias("RecordKey"),
    lit("bz_SalesCreditMemos").alias("TableName"),
    lit("id").alias("ColumnName"),
    lit("Null or Blank primary key 'id' identified").alias("ExceptionDetails"),
    current_date().alias("SysCreatedDate"),
    lit("databricks_job").alias("SysCreatedBy")
)

# Union and append exceptions to your isolated logging store
final_exceptions_df = duplicate_exceptions.unionByName(null_exceptions)
final_exceptions_df.cache()

if final_exceptions_df.count() > 0:
    final_exceptions_df.write.format("delta").mode("append").saveAsTable("erp_lakehouse.silver.exception_table")

final_exceptions_df.unpersist()

# Filter out exception records from the main execution branch
clean_result_df = result_df.filter(~col("RecordStatus").isin(['1', '2']))



In [0]:
# ==========================================
# 4. ENHANCED COLUMN MAPPING (SILVER SPECIFICATION)
# ==========================================
final_df = clean_result_df.select(
    col("CompanyKey").cast("string"),
    col("BrandKey").cast("string"),
    col("id").alias("InvSrcId").cast("string"),
    coalesce(col("salesperson"), lit("")).alias("SalesRepInSrcId").cast("string"),
    coalesce(col("customerNumber"), lit("")).alias("CustSrcId").cast("string"),
    coalesce(col("number"), lit("")).alias("InvKey").cast("string"),
    lit("").cast("string").alias("SOKey"),
    lit('CR').cast("string").alias("InvType"),
    lit('Credit Memo').cast("string").alias("InvTypeDesc"),
    coalesce(col("status"), lit("")).alias("InvProcessStatus").cast("string"),
    
    # Unified status logic mapping
    when(col("number") == "Open", lit("Open"))
        .when(col("status").isin("Cancelled", "Canceled"), lit("Cancelled"))
        .when(col("status").isin("Paid", "Corrective"), lit("Completed"))
        .otherwise(lit("Draft")).alias("InvStatus").cast("string"),
    lit("").cast("string").alias("InvPaidStatus"),
    
    # Date normalizations
    when(col("creditMemoDate") == "1900-01-01", lit(None)).otherwise(col("creditMemoDate")).alias("InvDt").cast("timestamp"),
    when(col("postingDate") == "1900-01-01", lit(None)).otherwise(col("postingDate")).alias("InvPostedDt").cast("timestamp"),
    when(col("dueDate") == "1900-01-01", lit(None)).otherwise(col("dueDate")).alias("InvDueDt").cast("timestamp"),
    datediff(col("dueDate"), col("creditMemoDate")).alias("InvDueDays").cast("int"),
    
    # Customer details
    coalesce(col("salesperson"), lit("")).alias("SalesRepInKey").cast("string"),
    coalesce(col("customerNumber"), lit("")).alias("CustKey").cast("string"),
    coalesce(col("customerName"), lit("")).alias("CustName").cast("string"),
    lit("").cast("string").alias("CustContactName"),
    
    # Bill-To mappings
    coalesce(col("billToCustomerNumber"), lit("")).alias("CustBillToAddrKey").cast("string"),
    coalesce(col("BillToName"), lit("")).alias("CustBillToName").cast("string"),
    coalesce(col("billToAddressLine1"), lit("")).alias("CustBillToAddr1").cast("string"),
    coalesce(col("billToAddressLine2"), lit("")).alias("CustBillToAddr2").cast("string"),
    coalesce(col("billToCity"), lit("")).alias("CustBillToCity").cast("string"),
    coalesce(col("billToState"), lit("")).alias("CustBillToState").cast("string"),
    coalesce(col("BilltoCountry"), lit("")).alias("CustBillToCountry").cast("string"),
    coalesce(col("billToPostCode"), lit("")).alias("CustBillToZipCd").cast("string"),
    
    # Clear blank placeholders for fields that don't belong to credit memos
    lit("").cast("string").alias("CustShipToName"),
    lit("").cast("string").alias("CustShipToAddr1"),
    lit("").cast("string").alias("CustShipToAddr2"),
    lit("").cast("string").alias("CustShipToCity"),
    lit("").cast("string").alias("CustShipToState"),
    lit("").cast("string").alias("CustShipToCountry"),
    lit("").cast("string").alias("CustShipToZipCd"),
    
    # Sell-To mappings
    coalesce(col("sellToAddressLine1"), lit("")).alias("CustSellToAddr1").cast("string"),
    coalesce(col("sellToAddressLine2"), lit("")).alias("CustSellToAddr2").cast("string"),
    coalesce(col("sellToCity"), lit("")).alias("CustSellToCity").cast("string"),
    coalesce(col("sellToState"), lit("")).alias("CustSellToState").cast("string"),
    coalesce(col("sellToCountry"), lit("")).alias("CustSellToCountry").cast("string"),
    coalesce(col("sellToPostCode"), lit("")).alias("CustSellToZipCd").cast("string"),
    
    # Financial indicators
    lit("INR").cast("string").alias("CustCurr"),
    lit("INR").cast("string").alias("LocCurr"),
    when(col("creditMemoDate") == "1900-01-01", lit(None)).otherwise(col("creditMemoDate")).alias("ExDt").cast("timestamp"),
    lit(1.0).cast("double").alias("ExRtLCCC"),
    lit(1.0).cast("double").alias("ExRtCCLC"),
    lit(1.0).cast("double").alias("ExRtLCIN"),
    
    # Core credit facts (multiplying by -1 to track balances accurately)
    when(col("TotalAmountExcludingTax").isNull(), lit(0.0)).otherwise(col("TotalAmountExcludingTax") * -1).cast("double").alias("TotalAmountExcludingTax"),
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValCC"),
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValLC"),
    coalesce(col("totalTaxAmount"), lit(0.0)).cast("double").alias("InvTaxValIN"),
    
    # Dynamic calculation fallbacks
    lit(0.0).cast("double").alias("InvPaidValCC"),
    lit(0.0).cast("double").alias("InvPaidValLC"),
    lit(0.0).cast("double").alias("InvPaidValIN"),
    lit(0.0).cast("double").alias("InvDueValCC"),
    lit(0.0).cast("double").alias("InvDueValLC"),
    lit(0.0).cast("double").alias("InvDueValIN"),
    
    # Tracking fields
    col("lastModifiedDateTime").alias("SourceUpdatedTime").cast("timestamp"),
    col("ingestion_time").alias("SysCreatedTime").cast("timestamp"),
    col("RecordStatus").cast("string")
)
display(final_df.limit(10))


In [0]:
# ==========================================
# 5. INLINE HIGH-PERFORMANCE MASTER REFS JOIN
# ==========================================
sales_rep_df = spark.read.table("erp_lakehouse.silver.sl_salesperson")

credits_df = final_df.join(
    sales_rep_df.select(col("SalesRepSrcId"), col("SalesRepName")),
    final_df["SalesRepInSrcId"] == sales_rep_df["SalesRepSrcId"],
    "left"
).withColumn(
    "SalesRepName", 
    coalesce(col("SalesRepName"), lit(""))
).drop("SalesRepSrcId")

print("Credit Memos pipeline processing successfully completed! 🎉")


## combining both credits and invoices

In [0]:
invoiceheader_df = invoice_df.unionByName(credits_df)

In [0]:
# ==========================================
# 6. EXTERNAL DELTA LAKE MERGE OPERATION
# ==========================================

# Create table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}.sl_invoiceheader (
    CompanyKey STRING,
    BrandKey STRING,
    InvSrcId STRING,
    SalesRepInSrcId STRING,
    CustSrcId STRING,
    InvKey STRING,
    SOKey STRING,
    InvType STRING,
    InvTypeDesc STRING,
    InvProcessStatus STRING,
    InvStatus STRING,
    InvPaidStatus STRING,
    InvDt TIMESTAMP,
    InvPostedDt TIMESTAMP,
    InvDueDt TIMESTAMP,
    InvDueDays INT,
    SalesRepInKey STRING,
    CustKey STRING,
    CustName STRING,
    CustContactName STRING,
    CustBillToAddrKey STRING,
    CustBillToName STRING,
    CustBillToAddr1 STRING,
    CustBillToAddr2 STRING,
    CustBillToCity STRING,
    CustBillToState STRING,
    CustBillToCountry STRING,
    CustBillToZipCd STRING,
    CustShipToName STRING,
    CustShipToAddr1 STRING,
    CustShipToAddr2 STRING,
    CustShipToCity STRING,
    CustShipToState STRING,
    CustShipToCountry STRING,
    CustShipToZipCd STRING,
    CustSellToAddr1 STRING,
    CustSellToAddr2 STRING,
    CustSellToCity STRING,
    CustSellToState STRING,
    CustSellToCountry STRING,
    CustSellToZipCd STRING,
    CustCurr STRING,
    LocCurr STRING,
    ExDt TIMESTAMP,
    ExRtLCCC DOUBLE,
    ExRtCCLC DOUBLE,
    ExRtLCIN DOUBLE,
    TotalAmountExcludingTax DOUBLE,
    InvTaxValCC DOUBLE,
    InvTaxValLC DOUBLE,
    InvTaxValIN DOUBLE,
    InvPaidValCC DOUBLE,
    InvPaidValLC DOUBLE,
    InvPaidValIN DOUBLE,
    InvDueValCC DOUBLE,
    InvDueValLC DOUBLE,
    InvDueValIN DOUBLE,
    SourceUpdatedTime TIMESTAMP,
    SysCreatedTime TIMESTAMP,
    RecordStatus STRING
)
USING DELTA
LOCATION '{SILVER_PATH}'
""")

In [0]:
# Create temp view for Credits_df
invoiceheader_df.createOrReplaceTempView("Invoices_temp_view")

spark.sql("""
MERGE INTO erp_lakehouse.silver.sl_InvoiceHeader AS tgt
USING Invoices_temp_view AS src
ON tgt.InvSrcId = src.InvSrcId AND tgt.CompanyKey = src.CompanyKey AND tgt.BrandKey = src.BrandKey
WHEN MATCHED THEN
  UPDATE SET
    tgt.SalesRepInSrcId = src.SalesRepInSrcId,
    tgt.CustSrcId = src.CustSrcId,
    tgt.InvProcessStatus = src.InvProcessStatus,
    tgt.InvStatus = src.InvStatus,
    tgt.InvPaidStatus = src.InvPaidStatus,
    tgt.InvPostedDt = src.InvPostedDt,
    tgt.InvDueDt = src.InvDueDt,
    tgt.SalesRepInKey = src.SalesRepInKey,
    tgt.CustName = src.CustName,
    tgt.CustContactName = src.CustContactName,
    tgt.CustBillToAddrKey = src.CustBillToAddrKey,
    tgt.CustBillToName = src.CustBillToName,
    tgt.CustBillToAddr1 = src.CustBillToAddr1,
    tgt.CustBillToAddr2 = src.CustBillToAddr2,
    tgt.CustBillToCity = src.CustBillToCity,
    tgt.CustBillToState = src.CustBillToState,
    tgt.CustBillToCountry = src.CustBillToCountry,
    tgt.CustBillToZipCd = src.CustBillToZipCd,
    tgt.CustShipToName = src.CustShipToName,
    tgt.CustShipToAddr1 = src.CustShipToAddr1,
    tgt.CustShipToAddr2 = src.CustShipToAddr2,
    tgt.CustShipToCity = src.CustShipToCity,
    tgt.CustShipToState = src.CustShipToState,
    tgt.CustShipToCountry = src.CustShipToCountry,
    tgt.CustShipToZipCd = src.CustShipToZipCd,
    tgt.CustSellToAddr1 = src.CustSellToAddr1,
    tgt.CustSellToAddr2 = src.CustSellToAddr2,
    tgt.CustSellToCity = src.CustSellToCity,
    tgt.CustSellToState = src.CustSellToState,
    tgt.CustSellToCountry = src.CustSellToCountry,
    tgt.CustSellToZipCd = src.CustSellToZipCd,
    tgt.ExDt = src.ExDt,
    tgt.ExRtLCCC = src.ExRtLCCC,
    tgt.ExRtCCLC = src.ExRtCCLC,
    tgt.ExRtLCIN = src.ExRtLCIN,
    tgt.TotalAmountExcludingTax=src.TotalAmountExcludingTax,
    tgt.InvTaxValCC = src.InvTaxValCC,
    tgt.InvTaxValLC = src.InvTaxValLC,
    tgt.InvTaxValIN = src.InvTaxValIN,
    tgt.InvPaidValCC = src.InvPaidValCC,
    tgt.InvPaidValLC = src.InvPaidValLC,
    tgt.InvPaidValIN = src.InvPaidValIN,
    tgt.InvDueValCC = src.InvDueValCC,
    tgt.InvDueValLC = src.InvDueValLC,
    tgt.InvDueValIN = src.InvDueValIN,
    tgt.SourceUpdatedTime = src.SourceUpdatedTime
WHEN NOT MATCHED THEN
  INSERT *
""")

# Uncache data to free memory cluster resources
processed_df.unpersist()
print("Pipeline executed successfully and cleanly.")